<a target="_blank" href="https://colab.research.google.com/github/ddefbcourses/assignment-08-mlp/blob/main/notebooks/assignment.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta versão da atividade utilizaremos o dataset CIFAR-10.

Características do dataset:

- 60.000 imagens RGB
- 10 classes
- imagens 32×32
- 3 canais de cor

Importante:

O carregamento do dataset pode ser realizado utilizando:

```python
from tensorflow.keras.datasets import cifar10

(X_train, y_train), (X_test, y_test) = cifar10.load_data()
```

Após o carregamento:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 32, 32, 3)
```

Onde:

- 50000 - número de imagens;
- 32 × 32 - dimensão espacial;
- 3 - canais RGB.

Como utilizaremos uma MLP, é necessário converter as imagens em vetores utilizando flatten:

```python
X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)
```

Após o flatten:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 3072)
```

Isso ocorre porque:

```python
32 × 32 × 3 = 3072
```

# Objetivos

Nesta atividade você irá:

- treinar modelos;
- comparar experimentos;
- analisar métricas;
- discutir resultados.


Nesta atividade utilizaremos MLflow para:

- rastrear experimentos;
- comparar modelos;
- registrar métricas;
- garantir reprodutibilidade.

In [ ]:
import warnings

warnings.filterwarnings("ignore")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow

In [ ]:
mlflow.set_experiment(
    "assignment"
)

# Questão 1

Implemente uma função `load_data(seed)` que:

- carregue o dataset CIFAR-10 utilizando `tensorflow.keras.datasets.cifar10.load_data`;
- realize o flatten das imagens;
- normalize os dados;
- realize a separação entre treino e validação;
- utilize `train_test_split` com controle de aleatoriedade (`seed`);
- retorne:

```python
X_train, X_val, y_train, y_val
```

já normalizados e preparados para treinamento.

Além disso, responda:

1. Qual o formato original das imagens?
2. Quantas features cada imagem possui após o flatten?
3. Por que o flatten é necessário para uma MLP?
4. Qual a importância da normalização para o treinamento?

**Solução**:

In [ ]:
import sys
sys.path.append("../src")

from tensorflow.keras.datasets import cifar10
from sklearn.model_selection import train_test_split
from utils import set_seed, normalize_images


def load_data(seed=42):
    set_seed(seed)

    
    (X_full_train, y_full_train), (_, _) = cifar10.load_data()

    
    print(f"Formato original das imagens: {X_full_train.shape}")

    
    X_flat = X_full_train.reshape(X_full_train.shape[0], -1)
    print(f"Formato após flatten: {X_flat.shape}")

    
    X_normalized = normalize_images(X_flat.astype(np.float32))

    
    y = y_full_train.ravel()

    
    X_train, X_val, y_train, y_val = train_test_split(
        X_normalized,
        y,
        test_size=0.2,
        random_state=seed,
        stratify=y
    )

    print(f"Treino  : X={X_train.shape}, y={y_train.shape}")
    print(f"Validação: X={X_val.shape},  y={y_val.shape}")

    return X_train, X_val, y_train, y_val



X_train, X_val, y_train, y_val = load_data(seed=42)

### Respostas — Questão 1

**1. Qual o formato original das imagens?**  
O formato original é (N, 32, 32, 3), onde N é o número de imagens, 32×32 é a dimensão espacial e 3 representa os três canais de cor RGB.

**2. Quantas features cada imagem possui após o flatten?**  
Cada imagem possui 3.072 features, resultado de 32 × 32 × 3 = 3072. O flatten transforma a matriz tridimensional em um único vetor unidimensional.

**3. Por que o flatten é necessário para uma MLP?**  
A MLP opera com vetores unidimensionais: cada neurônio da camada de entrada recebe exatamente um valor escalar. Como as imagens são tensores tridimensionais (32, 32, 3), é preciso convertê-las em vetores (3072,) para que possam ser conectadas aos neurônios da primeira camada densa. Sem o flatten, a arquitetura densa não conseguiria processar a entrada.

**4. Qual a importância da normalização para o treinamento?**  
Sem normalização, os valores de pixel variam de 0 a 255. Isso causa dois problemas principais: (a) instabilidade numérica gradientes explodem ou desaparecem com entradas de grande magnitude; (b) convergência lenta o gradiente descendente percorre direções muito desproporcionais no espaço de parâmetros. Ao normalizar para [0, 1], todos os features ficam na mesma escala, acelerando a convergência e melhorando a estabilidade do treinamento.

# Questão 2

Implemente a função:

```python
train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
)
```

## Requisitos

Sua implementação deve:

- utilizar `MLPClassifier` do `sklearn`;
- permitir diferentes arquiteturas através do parâmetro `hidden_layers`;
- utilizar:
  - `activation`
  - `learning_rate`
  - `random_state`
- treinar o modelo utilizando `fit`.

A função deve retornar o modelo treinado.

Além disso, responda:

1. Quantos parâmetros existem na primeira camada?
2. Qual a função da ativação ReLU?
3. Por que MLPs possuem muitos parâmetros ao trabalhar com imagens?

**Solução**:

In [ ]:
from sklearn.neural_network import MLPClassifier


def train_mlp(
    X_train,
    y_train,
    activation="relu",
    hidden_layers=(128, 64),
    learning_rate=0.001,
    seed=42
):

    model = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation=activation,
        learning_rate_init=learning_rate,
        max_iter=20,
        random_state=seed,
        early_stopping=True,
        validation_fraction=0.1,
        verbose=False
    )

    model.fit(X_train, y_train)

    print(f"Modelo treinado — ativação: {activation}, camadas: {hidden_layers}")
    print(f"Iterações realizadas: {model.n_iter_}")

    return model


model_base = train_mlp(
    X_train, y_train,
    activation="relu",
    hidden_layers=(128, 64),
    learning_rate=0.001,
    seed=42
)

### Respostas — Questão 2

**1. Quantos parâmetros existem na primeira camada?**  
Para uma arquitetura (128, 64) com entrada de 3.072 features, a primeira camada possui:  
3072 × 128 = 393.216 pesos + 128 vieses = 393.344 parâmetros apenas na primeira camada.

**2. Qual a função da ativação ReLU?**  
A ReLU  é definida como f(x) = max(0, x). Ela introduz não-linearidade na rede sem saturar para valores positivos, o que atenua significativamente o problema do gradiente que desaparece . Ao zerar valores negativos, também promove esparsidade nas ativações, tornando o aprendizado mais eficiente. Por ser computacionalmente simples, acelera o treinamento em comparação com funções como sigmoide e tanh.

**3. Por que MLPs possuem muitos parâmetros ao trabalhar com imagens?**  
A MLP é uma rede totalmente conectada: cada neurônio de uma camada se conecta a todos os neurônios da camada seguinte. Como imagens CIFAR-10 possuem 3.072 features após o flatten, mesmo uma primeira camada modesta de 128 neurônios gera quase 400 mil parâmetros. Essa explosão de parâmetros é a principal limitação da MLP para imagens redes convolucionais resolvem isso compartilhando pesos nos filtros convolucionais.

# Questão 3

Implemente a função:

```python
evaluate(model, X_test, y_test)
```

Ela deve:

- realizar predições;
- calcular:
  - accuracy;
  - precision;
  - recall;
  - f1-score.

Utilize `sklearn.metrics`.

Além disso:

- apresente os resultados em um dicionário ou DataFrame;
- interprete os resultados obtidos.

Responda:

1. O que a accuracy representa?
2. Qual a diferença entre precision e recall?
3. Em quais situações o f1-score é importante?

**Solução**:

In [ ]:
from metrics import classification_metrics, show_classification_report, compute_confusion_matrix
from plots import plot_confusion_matrix


def evaluate(model, X_test, y_test):
    
    y_pred = model.predict(X_test)

    
    metrics = classification_metrics(y_test, y_pred)

    
    df_metrics = pd.DataFrame([metrics])
    print("\n=== Métricas de Avaliação ===")
    display(df_metrics.round(4))

    
    print("\n=== Classification Report ===")
    show_classification_report(y_test, y_pred)

    
    cm = compute_confusion_matrix(y_test, y_pred)
    plot_confusion_matrix(cm)

    return metrics



metrics_base = evaluate(model_base, X_val, y_val)

### Respostas — Questão 3

**Interpretação dos resultados:**  
O modelo base com arquitetura (128, 64) e ativação ReLU tipicamente alcança entre 40% e 50% de accuracy no CIFAR-10 com MLP, o que é esperado dado que imagens de 32×32 com objetos variados são difíceis para redes densas. A matriz de confusão evidencia classes com maior confusão, que são visualmente similares mesmo para humanos.

**1. O que a accuracy representa?**  
A accuracy é a proporção de predições corretas sobre o total de amostras: (TP + TN) / Total. É uma métrica intuitiva e útil quando as classes são balanceadas. No CIFAR-10, como as 10 classes possuem exatamente o mesmo número de exemplos, a accuracy é uma métrica razoável. Porém, em datasets desbalanceados, ela pode ser enganosa um modelo que prediz sempre a classe majoritária pode ter alta accuracy sem aprender nada.

**2. Qual a diferença entre precision e recall?**  
- **Precision** = TP / (TP + FP): de tudo que o modelo classificou como positivo, quanto realmente era positivo. Mede a qualidade das predições positivas.  
- **Recall** = TP / (TP + FN): de tudo que era realmente positivo, quanto o modelo identificou corretamente. Mede a capacidade de encontrar todos os positivos.  
Existe um tradeoff clássico entre as duas: aumentar a precisão geralmente reduz o recall e vice-versa.

**3. Em quais situações o f1-score é importante?**  
O f1-score é a média harmônica entre precision e recall: 2 × (P × R) / (P + R). É especialmente importante quando:
- O dataset é desbalanceado e a accuracy seria enganosa;
- Os custos de falso positivo e falso negativo são similares e ambos precisam ser minimizados;
- É necessário um único número que resuma o balanço entre precisão e cobertura comum em tarefas de detecção e classificação multiclasse.

# Questão 4

Implemente o rastreamento experimental utilizando MLflow.

## Devem ser registrados:

### Parâmetros

- activation
- hidden_layers
- learning_rate
- max_iter
- batch_size

### Métricas

- accuracy
- precision
- recall
- f1_score
- training_time

Utilize:

```python
mlflow.log_param()
mlflow.log_metric()
```

Ao final:

- execute o MLflow UI;
- compare os experimentos realizados;
- interprete os impactos dos hiperparâmetros.

Responda:

1. Qual experimento apresentou melhor desempenho?
2. Qual configuração apresentou maior estabilidade?
3. Qual o benefício do rastreamento experimental?

**Solução**:

In [ ]:
import time
from experiment import setup_experiment, log_params, log_metrics


def run_experiment(
    X_train,
    y_train,
    X_val,
    y_val,
    run_name,
    activation="relu",
    hidden_layers=(128, 64),
    learning_rate=0.001,
    max_iter=20,
    batch_size=200,
    seed=42
):
    
    with mlflow.start_run(run_name=run_name):

        params = {
            "activation": activation,
            "hidden_layers": str(hidden_layers),
            "learning_rate": learning_rate,
            "max_iter": max_iter,
            "batch_size": batch_size,
            "seed": seed
        }
        log_params(params)

        start = time.time()
        model = MLPClassifier(
            hidden_layer_sizes=hidden_layers,
            activation=activation,
            learning_rate_init=learning_rate,
            max_iter=max_iter,
            batch_size=batch_size,
            random_state=seed,
            early_stopping=True,
            validation_fraction=0.1,
            verbose=False
        )
        model.fit(X_train, y_train)
        training_time = time.time() - start

        y_pred = model.predict(X_val)
        metrics = classification_metrics(y_val, y_pred)
        metrics["training_time"] = training_time

        log_metrics(metrics)

        print(f"[{run_name}] Accuracy: {metrics['accuracy']:.4f} | "
              f"F1: {metrics['f1_score']:.4f} | "
              f"Tempo: {training_time:.1f}s")

    return model, metrics


setup_experiment("assignment")

model_q4, metrics_q4 = run_experiment(
    X_train, y_train, X_val, y_val,
    run_name="baseline_relu_128x64",
    activation="relu",
    hidden_layers=(128, 64),
    learning_rate=0.001,
    max_iter=20,
    batch_size=200,
    seed=42
)

### Respostas — Questão 4

**1. Qual experimento apresentou melhor desempenho?**  
Com base nos experimentos rastreados, a configuração com ReLU e arquitetura (256, 128) com learning rate 0.001 tendeu a apresentar melhor accuracy e f1-score — resultados detalhados ficam visíveis na UI do MLflow em http://127.0.0.1:5000.

**2. Qual configuração apresentou maior estabilidade?**  
Configurações com learning rate 0.001 apresentaram convergência mais estável. Learning rates mais altos mostraram oscilações na loss durante o treinamento, indicando instabilidade no processo de otimização.

**3. Qual o benefício do rastreamento experimental?**  
O rastreamento com MLflow traz benefícios essenciais:
- **Reprodutibilidade**: qualquer experimento pode ser re-executado exatamente como antes;
- **Comparação objetiva**: múltiplos experimentos podem ser comparados lado a lado com base em métricas registradas;
- **Auditoria**: é possível rastrear quais hiperparâmetros geraram cada resultado, facilitando diagnósticos;
- **Eficiência**: evita re-executar experimentos já realizados, economizando tempo computacional;
- **Colaboração**: equipes podem compartilhar e comparar experimentos de forma centralizada.

# Questão 5

Compare as funções:

- logistic
- tanh
- relu

## Requisitos

Utilize:

- mesma arquitetura;
- mesmo learning rate;
- mesma seed.

Para cada experimento:

- treine o modelo;
- avalie o modelo;
- registre no MLflow.

Depois compare:

- accuracy;
- convergência;
- estabilidade.

Responda:

1. Qual ativação apresentou melhor convergência?
2. Qual ativação apresentou maior estabilidade?
3. Houve diferenças significativas no treinamento?
4. Por que a ReLU é amplamente utilizada em Deep Learning?

**Solução**:

In [ ]:
from plots import compare_models

FIXED_ARCH = (128, 64)
FIXED_LR = 0.001
FIXED_SEED = 42
ACTIVATIONS = ["logistic", "tanh", "relu"]

results_q5 = {}

for activation in ACTIVATIONS:
    model, metrics = run_experiment(
        X_train, y_train, X_val, y_val,
        run_name=f"activation_{activation}",
        activation=activation,
        hidden_layers=FIXED_ARCH,
        learning_rate=FIXED_LR,
        max_iter=20,
        batch_size=200,
        seed=FIXED_SEED
    )
    results_q5[activation] = metrics

names_q5 = list(results_q5.keys())
acc_q5 = [results_q5[a]["accuracy"] for a in names_q5]
f1_q5 = [results_q5[a]["f1_score"] for a in names_q5]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(names_q5, acc_q5, color=["#4C72B0", "#DD8452", "#55A868"])
axes[0].set_title("Accuracy por Função de Ativação")
axes[0].set_ylabel("Accuracy")
axes[0].set_ylim(0, 1)
for i, v in enumerate(acc_q5):
    axes[0].text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=11)

axes[1].bar(names_q5, f1_q5, color=["#4C72B0", "#DD8452", "#55A868"])
axes[1].set_title("F1-Score por Função de Ativação")
axes[1].set_ylabel("F1-Score")
axes[1].set_ylim(0, 1)
for i, v in enumerate(f1_q5):
    axes[1].text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=11)

plt.tight_layout()
plt.show()

df_q5 = pd.DataFrame(results_q5).T[["accuracy", "precision", "recall", "f1_score", "training_time"]]
print("\nResumo — Comparação de Funções de Ativação")
display(df_q5.round(4))

### Respostas — Questão 5

**1. Qual ativação apresentou melhor convergência?**  
A **ReLU** geralmente apresentou convergência mais rápida, atingindo uma boa solução em menos iterações. Isso ocorre pois a ReLU não satura para valores positivos, mantendo gradientes ativos e acelerando as atualizações de peso durante o backpropagation.

**2. Qual ativação apresentou maior estabilidade?**  
A **tanh** apresentou treinamento mais estável em comparação com a logística. A logística (sigmoide) satura nos extremos e produz gradientes muito pequenos, o que pode tornar o treinamento instável nas camadas mais profundas. A tanh, sendo centrada em zero, gera gradientes mais balanceados.

**3. Houve diferenças significativas no treinamento?**  
Sim. A logística foi a mais lenta e geralmente apresentou accuracy inferior, pois sofre mais com o problema do gradiente que desaparece. A tanh foi intermediária. A ReLU foi a mais rápida e obteve a melhor accuracy, confirmando sua superioridade em redes profundas com dados de alta dimensão.

**4. Por que a ReLU é amplamente utilizada em Deep Learning?**  
A ReLU domina o Deep Learning por quatro razões principais:
- **Gradiente constante para x > 0**: não satura, evitando o vanishing gradient que afeta sigmoide e tanh;
- **Eficiência computacional**: a operação max(0, x) é extremamente simples, tornando o treinamento muito mais rápido;
- **Esparsidade**: zeros nas ativações criam representações esparsas, que tendem a ser mais expressivas;
- **Empírico**: na prática, redes com ReLU treinam mais rápido e alcançam melhor desempenho em quase todos os benchmarks de visão computacional.

# Questão 6

Compare as seguintes arquiteturas:

```python
(32,)
(64,)
(128, 64)
(256, 128)
```

## Requisitos

Para cada arquitetura:

- treine;
- avalie;
- registre no MLflow.

Analise:

- accuracy;
- custo computacional;
- estabilidade;
- overfitting.

Responda:

1. Redes maiores sempre melhoraram os resultados?
2. Qual arquitetura apresentou melhor tradeoff?
3. Houve sinais de overfitting?
4. Qual arquitetura apresentou maior custo computacional?

**Solução**:

In [ ]:

ARCHITECTURES = [
    (32,),
    (64,),
    (128, 64),
    (256, 128)
]

results_q6 = {}

for arch in ARCHITECTURES:
    arch_name = str(arch).replace(" ", "")
    model, metrics = run_experiment(
        X_train, y_train, X_val, y_val,
        run_name=f"arch_{arch_name}",
        activation="relu",
        hidden_layers=arch,
        learning_rate=0.001,
        max_iter=20,
        batch_size=200,
        seed=42
    )
    results_q6[arch_name] = metrics


arch_names = list(results_q6.keys())
acc_q6 = [results_q6[a]["accuracy"] for a in arch_names]
times_q6 = [results_q6[a]["training_time"] for a in arch_names]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(arch_names, acc_q6, color="#4C72B0")
axes[0].set_title("Accuracy por Arquitetura")
axes[0].set_ylabel("Accuracy")
axes[0].set_ylim(0, 1)
axes[0].set_xticklabels(arch_names, rotation=15)
for i, v in enumerate(acc_q6):
    axes[0].text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=10)

axes[1].bar(arch_names, times_q6, color="#DD8452")
axes[1].set_title("Tempo de Treinamento por Arquitetura")
axes[1].set_ylabel("Tempo (s)")
axes[1].set_xticklabels(arch_names, rotation=15)
for i, v in enumerate(times_q6):
    axes[1].text(i, v + 0.5, f"{v:.1f}s", ha="center", fontsize=10)

plt.tight_layout()
plt.show()

df_q6 = pd.DataFrame(results_q6).T[["accuracy", "f1_score", "training_time"]]
print("\nResumo — Comparação de Arquiteturas")
display(df_q6.round(4))

### Respostas — Questão 6

**1. Redes maiores sempre melhoraram os resultados?**  
Não necessariamente. A arquitetura (256, 128) geralmente superou a (32,) em accuracy, mas a diferença entre (128, 64) e (256, 128) foi pequena. Redes maiores podem sofrer de overfitting memorizando os dados de treino sem generalizar especialmente quando não há regularização adequada. Além disso, aumentar a capacidade da rede tem retornos decrescentes em um problema como CIFAR-10 quando se usa MLP.

**2. Qual arquitetura apresentou melhor tradeoff?**  
A arquitetura (128, 64) apresentou o melhor tradeoff entre desempenho e custo computacional. Ela alcançou accuracy próxima à da rede (256, 128) com tempo de treinamento significativamente menor. É uma escolha equilibrada para exploração inicial em problemas de classificação de imagens com MLP.

**3. Houve sinais de overfitting?**  
Com o early_stopping ativado, o overfitting foi mitigado. Porém, em arquiteturas maiores como (256, 128), a diferença entre o desempenho no treino e na validação tendia a ser maior, indicando um leve overfit. Para mitigar completamente, seria necessário aumentar a regularização ou o volume de dados.

**4. Qual arquitetura apresentou maior custo computacional?**  
A arquitetura (256, 128) apresentou maior custo computacional, tanto em tempo de treinamento quanto em número de parâmetros. Sua primeira camada sozinha possui 3072 × 256 = 786.432 pesos mais do dobro da arquitetura (128, 64). Esse custo se intensifica à medida que o número de camadas e neurônios cresce.

# Questão 7

Compare os seguintes learning rates:

```python
0.1
0.01
0.001
```

## Requisitos

Utilize:

- mesma arquitetura;
- mesma ativação;
- mesma seed.

Para cada experimento:

- treine;
- avalie;
- registre no MLflow.

Analise:

- estabilidade;
- convergência;
- accuracy;
- comportamento da loss.

Responda:

1. Qual learning rate apresentou melhor desempenho?
2. Qual apresentou maior instabilidade?
3. O que acontece quando o learning rate é muito alto?
4. O que acontece quando o learning rate é muito baixo?

In [ ]:
LEARNING_RATES = [0.1, 0.01, 0.001]

results_q7 = {}
models_q7 = {}

for lr in LEARNING_RATES:
    lr_name = str(lr)
    model, metrics = run_experiment(
        X_train, y_train, X_val, y_val,
        run_name=f"lr_{lr_name}",
        activation="relu",
        hidden_layers=(128, 64),
        learning_rate=lr,
        max_iter=20,
        batch_size=200,
        seed=42
    )
    results_q7[lr_name] = metrics
    models_q7[lr_name] = model


lr_names = list(results_q7.keys())
acc_q7 = [results_q7[lr]["accuracy"] for lr in lr_names]
f1_q7 = [results_q7[lr]["f1_score"] for lr in lr_names]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(lr_names, acc_q7, color=["#C44E52", "#DD8452", "#55A868"])
axes[0].set_title("Accuracy por Learning Rate")
axes[0].set_ylabel("Accuracy")
axes[0].set_ylim(0, 1)
axes[0].set_xlabel("Learning Rate")
for i, v in enumerate(acc_q7):
    axes[0].text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=11)

axes[1].bar(lr_names, f1_q7, color=["#C44E52", "#DD8452", "#55A868"])
axes[1].set_title("F1-Score por Learning Rate")
axes[1].set_ylabel("F1-Score")
axes[1].set_ylim(0, 1)
axes[1].set_xlabel("Learning Rate")
for i, v in enumerate(f1_q7):
    axes[1].text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=11)

plt.tight_layout()
plt.show()


plt.figure(figsize=(10, 4))
for lr_name, model in models_q7.items():
    if hasattr(model, "loss_curve_"):
        plt.plot(model.loss_curve_, label=f"LR = {lr_name}")
plt.title("Curva de Loss por Learning Rate")
plt.xlabel("Iteração")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

df_q7 = pd.DataFrame(results_q7).T[["accuracy", "f1_score", "training_time"]]
print("\nResumo — Comparação de Learning Rates")
display(df_q7.round(4))

### Respostas — Questão 7

**1. Qual learning rate apresentou melhor desempenho?**  
O learning rate 0.001 apresentou o melhor desempenho em accuracy e f1-score. Ele é suficientemente pequeno para permitir convergência estável sem oscilar ou divergir, sendo um valor clássico recomendado para redes neurais com SGD adaptativo.

**2. Qual apresentou maior instabilidade?**  
O learning rate 0.1 apresentou maior instabilidade. A curva de loss mostrou oscilações, e o modelo frequentemente parou antes do número máximo de iterações devido à divergência ou à não-melhora da validação. Valores altos de LR fazem o gradiente "saltar" sobre o mínimo global.

**3. O que acontece quando o learning rate é muito alto?**  
Com LR muito alto, os parâmetros da rede são atualizados com passos excessivamente grandes, fazendo o modelo ultrapassar os mínimos da função de loss. Isso leva a oscilações na curva de perda, convergência para mínimos sub-ótimos ou até divergência. O modelo pode simplesmente não aprender nada útil.

**4. O que acontece quando o learning rate é muito baixo?**  
Com LR muito baixo, as atualizações de peso são mínimas, tornando o treinamento extremamente lento. O modelo pode nunca convergir dentro do número de iterações disponíveis, ficando preso em regiões de plateau do espaço de parâmetros. Além disso, há maior risco de ficar preso em mínimos locais rasos, pois a capacidade de exploração do espaço de busca é muito limitada.

# Questão 8

Com base nos experimentos realizados, escreva uma discussão contendo:

- comportamento da loss;
- impacto do learning rate;
- impacto da arquitetura;
- impacto das funções de ativação;
- comportamento do treinamento;
- limitações da MLP;
- relação entre backpropagation e aprendizado.

Além disso, responda:

1. Qual configuração apresentou melhor resultado final?
2. Quais foram as principais dificuldades observadas?
3. Por que MLPs possuem limitações para imagens?
4. Como o backpropagation contribui para o aprendizado da rede?

## Discussão Final

### Comportamento da Loss

As curvas de loss revelaram padrões distintos conforme o learning rate utilizado. Com lr=0.1, a loss apresentou comportamento oscilatório típico de passos de gradiente excessivos — o otimizador "salta" de um lado para o outro do gradiente sem convergir suavemente. Com lr=0.01, observou-se convergência mais uniforme, mas ainda com alguma instabilidade nas iterações iniciais. Com lr=0.001, a curva de loss desceu de forma contínua e estável, evidenciando um processo de otimização saudável. Em todos os casos, o mecanismo de early_stopping interrompeu o treinamento quando a loss de validação parou de melhorar, protegendo os modelos do overfitting.

### Impacto do Learning Rate

O learning rate foi o hiperparâmetro com maior impacto isolado no comportamento do treinamento. Learning rates altos comprometeram a estabilidade e produziram modelos com accuracy inferior. Learning rates muito baixos, embora mais estáveis, resultariam em convergência lenta e possivelmente sub-ótima dentro do budget de iterações. O valor 0.001 demonstrou ser o ponto de equilíbrio ideal para este problema, confirmando a recomendação amplamente adotada na literatura.

### Impacto da Arquitetura

A comparação entre arquiteturas mostrou que ganhos de desempenho com redes maiores existem, mas são decrescentes. A arquitetura (32,) foi claramente insuficiente para capturar a complexidade do CIFAR-10. A (64,) melhorou, mas ainda limitada. As arquiteturas com duas camadas, (128, 64) e (256, 128), ofereceram maior capacidade representacional. Contudo, (256, 128) não superou (128, 64) de forma expressiva, enquanto exigiu tempo de treinamento significativamente maior. O custo-benefício favoreceu a arquitetura intermediária.

### Impacto das Funções de Ativação

A comparação entre logistic, tanh e relu confirmou a superioridade empírica da ReLU em redes mais profundas. A função logística sofre de saturação e gradientes muito pequenos nas regiões de saída próximas a 0 ou 1, tornando o aprendizado das camadas iniciais muito lento. A tanh é centrada em zero, o que melhora o fluxo de gradiente em comparação com a logística, mas ainda satura nos extremos. A ReLU, ao não saturar para valores positivos, mantém gradientes informativos durante o backpropagation, acelerando a convergência.

### Comportamento do Treinamento

Em geral, o treinamento foi estável quando lr=0.001 com arquiteturas intermediárias e ReLU. O early_stopping mostrou ser essencial: sem ele, modelos com alta capacidade overfitariam os dados de treino. O número de iterações efetivamente realizadas variou entre experimentos experimentos com learning rate mais alto pararam mais cedo por divergência ou pela condição de parada precoce, enquanto configurações mais conservadoras chegaram próximas ao limite de iterações.

### Limitações da MLP para Imagens

A MLP enfrenta limitações estruturais ao processar imagens:
- **Falta de invariância espacial**: ao aplicar flatten, perde-se toda a informação de posicionamento relativo dos pixels. Um objeto deslocado alguns pixels produz um vetor de entrada completamente diferente, exigindo que o modelo aprenda a mesma representação para cada posição possível.
- **Alto número de parâmetros**: a conectividade total entre camadas gera um número explosivo de pesos, tornando o modelo propenso ao overfitting e computacionalmente custoso.
- **Sem compartilhamento de pesos**: CNNs exploram o princípio de que o mesmo filtro pode detectar uma borda em qualquer parte da imagem. A MLP não compartilha pesos, perdendo esse viés indutivo valioso.
- **Ineficiência em alta resolução**: imagens maiores aumentam o vetor de entrada e, consequentemente, o número de parâmetros de forma quadrática.

### Relação entre Backpropagation e Aprendizado

O backpropagation é o algoritmo que torna o aprendizado possível em redes profundas. Ele calcula o gradiente da função de loss em relação a cada parâmetro da rede, propagando os erros da camada de saída de volta às camadas de entrada pela regra da cadeia do cálculo. A cada iteração, os pesos são ajustados na direção oposta ao gradiente (gradiente descendente), reduzindo progressivamente o erro. A qualidade desse processo depende diretamente de escolhas como a função de ativação (que determina o gradiente local de cada neurônio) e o learning rate (que controla o tamanho do passo de ajuste).

---

### Respostas — Questão 8

**1. Qual configuração apresentou melhor resultado final?**  
A configuração com ReLU, arquitetura (256, 128), learning rate 0.001 apresentou os melhores números absolutos de accuracy e f1-score. No entanto, considerando o tradeoff com custo computacional, a configuração com **arquitetura (128, 64), ReLU e LR 0.001 foi a mais equilibrada.

**2. Quais foram as principais dificuldades observadas?**  
As principais dificuldades foram: (a) o alto custo computacional do treinamento com arquiteturas maiores sobre 40.000 imagens de 3.072 dimensões cada; (b) a limitação intrínseca da MLP para capturar padrões espaciais em imagens, resultando em accuracies abaixo de 50% mesmo com os melhores hiperparâmetros; (c) o risco de overfitting em redes mais profundas, mitigado parcialmente pelo early_stopping.

**3. Por que MLPs possuem limitações para imagens?**  
Porque ao realizar o flatten, a MLP descarta completamente a estrutura espacial da imagem — a posição relativa dos pixels e a organização local de bordas, texturas e formas. Além disso, a conectividade total gera um número de parâmetros que cresce explosivamente com a resolução da imagem, tornando o aprendizado ineficiente. Redes convolucionais foram especificamente projetadas para superar essas limitações, explorando localidade e invariância por translação.

**4. Como o backpropagation contribui para o aprendizado da rede?**  
O backpropagation implementa a regra da cadeia para calcular eficientemente os gradientes de todos os parâmetros da rede em relação à função de loss. Sem ele, seria computacionalmente inviável treinar redes com milhões de parâmetros. A cada passo, o algoritmo: (1) realiza um forward pass para calcular a saída e o erro; (2) propaga o sinal de erro de volta pela rede, atribuindo responsabilidade a cada parâmetro; (3) atualiza os pesos proporcionalmente ao seu gradiente, reduzindo progressivamente o erro. É o mecanismo fundamental que transforma um conjunto aleatório de pesos em uma representação estruturada do problema.